# Анализ продаж: Python → pandas

**Портфольный кейс BI-аналитика.** Данные: 11 000 записей за 2023–2024
(категории, регионы, сегменты, выручка, себестоимость, прибыль), из них
10 921 — фактические продажи (остальные отбракованы как дефектные).

Файл данных: `data/sales_2023_2024.csv` · Графики: `output/`

**Структура работы**

| Шаг | Содержание |
|---|---|
| 0 | Загрузка и типы данных |
| 1 | Уборка данных + контроль качества |
| 2 | Агрегация, KPI, ABC-анализ, вклад в рост |
| 3 | Визуализация (9 графиков) |
| 4 | Выводы для бизнеса |

Все числовые подписи на графиках **вычисляются из данных**, а не вписаны вручную:
если данные изменятся, текст обновится сам и не сможет разойтись с расчётом.


In [1]:
# --- импорты и настройки ---
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Шрифт: сначала системный, при отсутствии — встроенный (кириллица есть в обоих)
plt.rcParams["font.family"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

GREEN, RED, GRAY, BLUE = "#2e7d32", "#c62828", "#9e9e9e", "#4C72B0"

# --- пути: относительные, чтобы ноутбук запускался из любой папки ---
BASE = Path.cwd()
DATA_FILE = BASE / "data" / "sales_2023_2024.csv"
SAVE_DIR = BASE / "output"
SAVE_DIR.mkdir(exist_ok=True)

print("Данные:", DATA_FILE.relative_to(BASE), "| найден:", DATA_FILE.exists())
print("Графики ->", SAVE_DIR.relative_to(BASE))

Данные: data\sales_2023_2024.csv | найден: True
Графики -> output


## Шаг 0. Загрузка и первый взгляд

1. Читаем CSV с `encoding="utf-8-sig"` — на случай BOM в выгрузке из Excel.
2. Смотрим `info()` / `describe()` / `head()` / `shape` — понимаем типы и объём **до** любых преобразований.
3. `order_date` приводим к `datetime` и добавляем производные колонки `year`, `quarter`, `month`.

Ключевой момент: **datetime приводим до любой группировки**, иначе получим
`sorted()` по строкам вместо хронологического порядка.


In [2]:
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")

print("Форма:", df.shape)
df.info()
df.head()

df["order_date"] = pd.to_datetime(df["order_date"], errors="raise")
df["year"]     = df["order_date"].dt.year.astype(int)      # int — тип больше не меняем
df["quarter"]  = df["order_date"].dt.quarter.astype(int)
df["month"]    = df["order_date"].dt.to_period("M")
df["ym"]       = df["order_date"].dt.strftime("%Y-%m")

print("\nДиапазон дат:", df["order_date"].min().date(), "—", df["order_date"].max().date())
print("Категорий:", df["category"].nunique(),
      "| регионов:", df["region"].nunique(),
      "| сегментов:", df["segment"].nunique())

assert df["order_date"].notna().all(), "Есть нераспознанные даты"
assert df["order_id"].is_unique, "order_id должен быть уникальным ключом"
print("\n✓ даты распознаны, order_id уникален")

Форма: (11000, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11000 entries, 0 to 10999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    11000 non-null  int64  
 1   order_date  11000 non-null  object 
 2   category    11000 non-null  object 
 3   region      11000 non-null  object 
 4   segment     11000 non-null  object 
 5   units       10835 non-null  float64
 6   price       11000 non-null  float64
 7   revenue     11000 non-null  float64
 8   cost        11000 non-null  float64
 9   profit      11000 non-null  float64
dtypes: float64(5), int64(1), object(4)
memory usage: 859.5+ KB

Диапазон дат: 2023-01-01 — 2024-12-31
Категорий: 5 | регионов: 5 | сегментов: 3

✓ даты распознаны, order_id уникален


## Шаг 1. Уборка данных (Data Cleaning)

В данных намеренно «вывалили грязь». Первое, что нужно сделать, — **не «чинить»,
а классифицировать**: разные дефекты требуют разных решений, и ошибка в
классификации даёт ошибку в метриках.

Структура дефектов:

| Тип | Признак | Число строк | Решение |
|---|---|---|---|
| Пропуск | `units = NaN` | 165 | заполнить медианой |
| Заглушка | `revenue = -1`, при этом `cost` и `profit` валидны | 110 | **восстановить** |
| Пустой заказ | `revenue = cost = profit = 0` | 78 | **исключить** из KPI |
| Дубли | `duplicated()` / `order_id` | 0 | — |

### 1.1 Пропуски
Заполняем `units` **медианой**, а не средним: количество заказанных единиц имеет
тяжёлый хвост (оптовые партии), и одно крупное значение сдвинуло бы среднее.
Медиана к такому выбросу устойчива.

### 1.2 `revenue = -1` — это не возврат, а **незаполненное поле**

Первая гипотеза «это возвраты» **не подтверждается**:

* если бы это был возврат, отрицательными должны были быть `revenue`, `cost` и количество;
* но `cost` положителен, `profit` положителен и **не равна** `revenue − cost`;
* у всех «грязных» строк значение ровно одно: `−1.0` (сумма = `−110.00` = `−1 × 110`).

Единичное значение `−1` — это **заглушка (sentinel)**, а не бизнес-событие.
Значит поле нужно **восстановить**, а не удалить и не «перевести в ноль».

Восстанавливаем **двумя независимыми формулами** и сверяем их:

```
revenue = units × price      и      revenue = cost + profit
```

Если обе дают одно и то же — значение восстановлено однозначно, и мы не гадаем.
В коде используем `cost + profit`, потому что он **не зависит от `units`**,
а это важно — см. п. 1.4.

### 1.3 Пустые заказы — внутренне непротиворечивые, но это не продажи

Отдельно ловим строки, где `revenue = 0`. Их **`cost` и `profit` тоже равны нулю**,
то есть `profit = revenue − cost` выполняется — данные согласованы, «чинить» нечего.

Но продажа нулевого количества единиц за ноль рублей — это не продажа. Наиболее
вероятная причина — усечение количества до целого: очень малые партии
(дробная величина меньше 1) стали нулевыми. И это напрямую вредит метрикам:

* в знаменатель числа заказов попадают 78 «нулевых» строк;
* при этом в выручку они добавляют ровно ноль;
* итог: средний чек занижается примерно на **11%**.

Поэтому их **исключаем из KPI**, но **не удаляем из датасета** — считаем и показываем
явно, вместе с оценкой влияния (чувствительность метрики).

### 1.4 Порядок операций (главная ловушка)

Часть строк имеет **одновременно** пропуск в `units` и заглушку в `revenue`.
Если сначала заполнить `units` медианой, а потом считать `revenue = units × price`,
то для этих строк выручка посчитается от **медианного** количества и метрики поедут.

Поэтому порядок такой: **сначала** восстанавливаем выручку из `cost + profit`
(единиц не требуется), **потом** заполняем пропуски в `units`.

### 1.5 Контроль качества

После уборки — проверки `assert`. Если хоть одна не проходит, пайплайн падает сразу,
а не отдаёт в отчётность испорченные цифры.


In [3]:
print("=== 1.1 Пропуски ===")
missing = df.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "пропусков нет")
print("строк с пропусками:", int(df.isna().any(axis=1).sum()))

print("\n=== 1.2 Тип 1: заглушка revenue = -1 ===")
sentinel = df["revenue"] < 0
print("строк с revenue < 0          :", int(sentinel.sum()))
print("уникальные значения revenue  :", sorted(df.loc[sentinel, "revenue"].unique()))
print("сумма revenue на них         : %.2f  (= -1 x %d)"
      % (df.loc[sentinel, "revenue"].sum(), int(sentinel.sum())))
print("из них units == 0            :", int((sentinel & (df["units"] == 0)).sum()))
print("cost < 0 :", int((df["cost"] < 0).sum()),
      "| profit < 0 :", int((df["profit"] < 0).sum()))
print("строк, где profit != revenue - cost:", int(((df["profit"] - (df["revenue"] - df["cost"])).abs() > 0.01).sum()))
print("  -> несогласованность возникает ИМЕННО на этих строках, значит revenue там не пересчитан")

# две независимые формулы восстановления — должны совпасть
rec_a = df.loc[sentinel, "units"] * df.loc[sentinel, "price"]    # units * price
rec_b = df.loc[sentinel, "cost"] + df.loc[sentinel, "profit"]    # cost + profit
delta = (rec_a - rec_b).abs()
print("\nвосстановление двумя формулами:")
print("  макс. расхождение            = %.6f" % delta.max())
print("  строк с расхождением > 0.01  = %d" % int((delta > 0.01).sum()))
print("  -> значение восстановимо однозначно")

print("\n=== 1.3 Тип 2: пустые заказы (revenue = 0) ===")
zero = df["revenue"] == 0
print("строк с revenue == 0     :", int(zero.sum()))
print("у них cost == 0          :", int((zero & (df["cost"] == 0)).sum()))
print("у них profit == 0        :", int((zero & (df["profit"] == 0)).sum()))
print("у них units == 0         :", int((zero & (df["units"] == 0)).sum()))
print("profit = revenue - cost выполняется на них:",
      bool(((df.loc[zero, "profit"] - (df.loc[zero, "revenue"] - df.loc[zero, "cost"])).abs()
            < 0.01).all()))
print("  -> данные согласованы, это не ошибка загрузки")
print("  -> вероятная причина: количество усекалось до целого, и очень малые партии")
print("     (дробное значение < 1) превратились в нулевые -> фактической продажи не было")
print("  -> в выручку дают 0, но в знаменатель числа заказов попадают -> вредят среднему чеку")

# ---- 1.4 ПОРЯДОК ОПЕРАЦИЙ: сначала revenue, потом units ----
rows_before = len(df)

df["revenue_recovered"] = sentinel          # флаг аудита: какие строки восстановили
df.loc[sentinel, "revenue"] = df.loc[sentinel, "cost"] + df.loc[sentinel, "profit"]
df.loc[sentinel, "profit"]  = df.loc[sentinel, "revenue"] - df.loc[sentinel, "cost"]

# теперь заполняем пропуски — они уже ни на что не влияют
df["units"] = df["units"].fillna(df["units"].median())

# пустые заказы помечаем, но НЕ удаляем из датасета
df["is_void"] = df["revenue"] == 0
df_void = df[df["is_void"]]
df_c = df[~df["is_void"]].copy()            # рабочий фрейм для всех метрик

print("\n=== 1.3 Дубли ===")
print("полных дублей   :", int(df.duplicated().sum()))
print("дублей order_id :", int(df["order_id"].duplicated().sum()))

print("\n=== Итог уборки ===")
print(f"строк до уборки            : {rows_before}")
print(f"строк после (датасет цел)  : {len(df)}   (удалено {rows_before - len(df)})")
print(f"восстановлено revenue      : {int(df['revenue_recovered'].sum())} строк")
print(f"заполнено units медианой   : {int(missing.get('units', 0))} строк")
print(f"помечено как пустой заказ  : {int(df['is_void'].sum())} строк")
print(f"строк в рабочем фрейме     : {len(df_c)}")

print("\n=== Чувствительность: влияние пустых заказов на средний чек ===")
rev_total = df["revenue"].sum()
print(f"выручка  с пустыми : {rev_total:>14,.2f} ₽  при {len(df):,} заказах "
      f"-> чек {rev_total/len(df):>8,.2f} ₽")
print(f"выручка без пустых : {df_c['revenue'].sum():>14,.2f} ₽  при {len(df_c):,} заказах "
      f"-> чек {df_c['revenue'].sum()/len(df_c):>8,.2f} ₽")
print(f"занижение среднего чека, если НЕ исключить: "
      f"{(1 - (rev_total/len(df)) / (df_c['revenue'].sum()/len(df_c)))*100:.1f}%")
print("-> решение: пустые заказы исключаем из KPI, но держим в датасете и считаем явно")

# ---- 1.5 КОНТРОЛЬ КАЧЕСТВА ----
assert (df["revenue"] >= 0).all(), "Остались отрицательные значения revenue"
assert df["units"].notna().all(), "Остались пропуски в units"
assert (df["profit"] - (df["revenue"] - df["cost"])).abs().max() < 0.01, \
    "profit не сходится с revenue - cost"
assert df["order_id"].is_unique, "Дубли order_id"
assert len(df) == rows_before, "Ни одна строка не должна исчезнуть при уборке"
assert (df_c["revenue"] > 0).all(), "В рабочем фрейме не должно быть нулевой выручки"
assert (df_c["units"] > 0).all(), "В рабочем фрейме не должно быть нулевого количества"
print("\n✓ control passed:")
print("   revenue >= 0 | units без пропусков | profit = revenue - cost |")
print("   order_id уникален | строки не теряются | в рабочем фрейме revenue > 0")

print(f"\nмедиана units (использовалась для заполнения): {df['units'].median():.1f}")
print(f"средний чек (рабочий фрейм): {df_c['revenue'].mean():,.2f} ₽")

# сохраняем счётчики уборки и дальше работаем ТОЛЬКО с рабочим фреймом
n_recovered    = int(df["revenue_recovered"].sum())
n_units_filled = int(missing.get("units", 0))
n_void         = int(df["is_void"].sum())
n_dupes        = int(df.duplicated().sum())
df = df_c
df.head(3)

=== 1.1 Пропуски ===
units    165
dtype: int64
строк с пропусками: 165

=== 1.2 Тип 1: заглушка revenue = -1 ===
строк с revenue < 0          : 110
уникальные значения revenue  : [np.float64(-1.0)]
сумма revenue на них         : -110.00  (= -1 x 110)
из них units == 0            : 1
cost < 0 : 0 | profit < 0 : 0
строк, где profit != revenue - cost: 110
  -> несогласованность возникает ИМЕННО на этих строках, значит revenue там не пересчитан

восстановление двумя формулами:
  макс. расхождение            = 0.000000
  строк с расхождением > 0.01  = 0
  -> значение восстановимо однозначно

=== 1.3 Тип 2: пустые заказы (revenue = 0) ===
строк с revenue == 0     : 78
у них cost == 0          : 78
у них profit == 0        : 78
у них units == 0         : 77
profit = revenue - cost выполняется на них: True
  -> данные согласованы, это не ошибка загрузки
  -> вероятная причина: количество усекалось до целого, и очень малые партии
     (дробное значение < 1) превратились в нулевые -> фактической

,order_id,order_date,category,region,segment,units,price,revenue,cost,profit,year,quarter,month,ym,revenue_recovered,is_void
0,1,2024-10-10,Техника,Москва,Потребитель,11.0,263.30,2896.30,1961.25,935.05,2024,4,2024-10,2024-10,False,False
1,2,2023-11-19,Канцелярия,Самара,Потребитель,13.0,437.75,5690.75,3491.08,2199.67,2023,4,2023-11,2023-11,False,False
2,3,2023-04-02,Офис,Екатеринбург,Потребитель,16.0,164.23,2627.68,1600.52,1027.16,2023,2,2023-04,2023-04,False,False


## Шаг 2. Агрегация, KPI, ABC и вклад в рост

Тот же набор расчётов, что и в SQL, — `groupby` = `GROUP BY`, `pivot_table` = сводная,
`.mean()` = среднее. Структура кода один в один повторяет привычный запрос.

### 2.1 Помесячные KPI
Выручка, прибыль, маржа, число заказов, средний чек.

### 2.2 Топ категорий по выручке и по прибыли
Сравниваем **масштаб** (выручка) и **эффективность** (прибыль). Если рейтинг по
двум метрикам различается — это сигнал о структуре затрат внутри категории.

### 2.3 Регион × сегмент
Сводная таблица — на неё смотрим как на источник данных для тепловой карты.

### 2.4 Год к году
Сравнение 2023 и 2024: сумма, средний чек, число заказов.

### 2.5 ABC-анализ — **две версии**
По 5 категориям ABC вырождается: класс «C» может содержать позицию с существенной
долей. Поэтому считаем и по категориям, и по связкам **категория × регион** (25 позиций) —
вторая версия статистически осмысленнее.

Конвенция: позиция идёт в класс, если **накопленная** доля на ней не превышает порог.
Это стандартная, но не единственная трактовка 80/15/5 — поэтому итоговые доли
сверяем с целевыми и фиксируем расхождение.

### 2.6 Вклад сегментов в прирост
`contribution = (выручка_2024 − выручка_2023) / общий прирост`. Сумма даёт 100%.

**Осторожно со знаком:** если общий итог отрицательный (а он отрицательный —
выручка упала), то вклад растущего сегмента тоже становится отрицательным.
Поэтому вклад полезен только рядом со столбцами `delta` в рублях и `pct` в процентах:
читать проценты вклада без знака итога — ошибка интерпретации.


In [4]:
# --- 2.1 помесячные KPI ---
monthly = (
    df.groupby("month")
      .agg(orders=("order_id", "count"),
           revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           avg_check=("revenue", "mean"))     # среднее revenue по заказам = сумма / count
)
monthly["margin"] = monthly["profit"] / monthly["revenue"] * 100
print("=== 2.1 Помесячные KPI (последние 6) ===")
print(monthly.tail(6).round(2).to_string())

# --- 2.2 топ категорий: масштаб vs эффективность ---
cats = (df.groupby("category")
          .agg(revenue=("revenue", "sum"), profit=("profit", "sum"),
               orders=("order_id", "count")))
cats["margin"] = cats["profit"] / cats["revenue"] * 100
top_by_rev  = cats.sort_values("revenue", ascending=False).head(3)
top_by_prof = cats.sort_values("profit",  ascending=False).head(3)
print("\n=== 2.2 Топ-3 по выручке (масштаб) ===")
print(top_by_rev[["revenue", "margin"]].round(2).to_string())
print("=== Топ-3 по прибыли (эффективность) ===")
print(top_by_prof[["profit", "margin"]].round(2).to_string())
print("совпадают ли рейтинги:",
      list(top_by_rev.index) == list(top_by_prof.index))

# --- 2.3 регион × сегмент ---
pivot = df.pivot_table(values="revenue", index="region", columns="segment",
                       aggfunc="sum")
print("\n=== 2.3 Регион × сегмент, выручка ===")
print((pivot / 1e6).round(2).to_string())

# --- 2.4 год к году ---
year_stats = df.groupby("year")["revenue"].agg(["sum", "mean", "count"])
year_stats.columns = ["revenue", "avg_check", "orders"]
year_stats["yoy_%"] = (year_stats["revenue"].pct_change() * 100).round(2)
print("\n=== 2.4 Год к году ===")
print(year_stats.round(2).to_string())

=== 2.1 Помесячные KPI (последние 6) ===
         orders     revenue      profit  avg_check  margin
month                                                     
2024-07     491  3576635.98  1212265.68    7284.39   33.89
2024-08     456  3235682.00  1075687.46    7095.79   33.24
2024-09     416  2893457.99   939149.14    6955.43   32.46
2024-10     435  3048989.35   978986.29    7009.17   32.11
2024-11     453  2900090.34   923059.41    6401.97   31.83
2024-12     447  3321088.19  1067876.09    7429.73   32.15

=== 2.2 Топ-3 по выручке (масштаб) ===
                revenue  margin
category                       
Дом и сад   21795885.92   32.43
Офис        16774575.17   32.06
Канцелярия  16002180.93   32.93
=== Топ-3 по прибыли (эффективность) ===
                profit  margin
category                      
Дом и сад   7069446.27   32.43
Офис        5377264.71   32.06
Канцелярия  5270154.46   32.93
совпадают ли рейтинги: True

=== 2.3 Регион × сегмент, выручка ===
segment          Корпора

In [5]:
# --- 2.5 ABC-анализ: две версии ---
def abc_table(revenue_by_item, thresholds=(80, 95)):
    """Присваивает классы A/B/C по накопленной доле выручки."""
    rev = revenue_by_item.sort_values(ascending=False)
    total = rev.sum()
    share = rev / total * 100
    cum = share.cumsum()
    a, b = thresholds
    cls = ["A" if c <= a else ("B" if c <= b else "C") for c in cum]
    out = pd.DataFrame({"revenue": rev, "share_%": share,
                        "cum_%": cum, "class": cls})
    actual = out.groupby("class")["share_%"].sum()
    target = pd.Series({"A": a, "B": b - a, "C": 100 - b})
    report = pd.DataFrame({"факт_%": actual.round(2), "цель_%": target})
    return out, report

# версия 1: по категориям (5 позиций)
abc_cat, rep_cat = abc_table(df.groupby("category")["revenue"].sum())
print("=== 2.5 ABC по категориям (5 позиций) ===")
print(abc_cat.round(2).to_string())
print(rep_cat.to_string())

# версия 2: по связкам категория × регион (25 позиций) — статистически осмысленнее
abc_cr, rep_cr = abc_table(df.groupby(["category", "region"])["revenue"].sum())
print("\n=== ABC по «категория × регион» (25 позиций) ===")
print("позиций в классах:", abc_cr["class"].value_counts().to_dict())
print(rep_cr.to_string())

# --- 2.6 вклад в прирост ---
seg_y = df.pivot_table(index="segment", columns="year", values="revenue", aggfunc="sum")
total_delta = (seg_y[2024].sum() - seg_y[2023].sum())
seg_y["delta"]   = seg_y[2024] - seg_y[2023]
seg_y["pct"]     = (seg_y["delta"] / seg_y[2023] * 100).round(2)
seg_y["contrib_%"] = (seg_y["delta"] / total_delta * 100).round(1)
print("\n=== 2.6 Вклад сегментов в изменение выручки 2023→2024 ===")
print(seg_y[[2023, 2024, "delta", "pct", "contrib_%"]].round(2).to_string())
print(f"общий прирост: {total_delta:,.0f} руб. | сумма вкладов: {seg_y['contrib_%'].sum():.1f}%")
print("ВНИМАНИЕ: общий итог отрицательный, поэтому знак 'contrib_%' инвертирован —")
print("растущий сегмент даёт отрицательный вклад. Читать вклад только рядом с delta/pct.")
print(f"Вклад по абсолютной величине (тыс. ₽): "
      + ", ".join(f"{k} {v/1e3:+,.0f}" for k, v in seg_y['delta'].sort_values().items()))

=== 2.5 ABC по категориям (5 позиций) ===
                revenue  share_%   cum_% class
category                                      
Дом и сад   21795885.92    27.95   27.95     A
Офис        16774575.17    21.51   49.46     A
Канцелярия  16002180.93    20.52   69.99     A
Техника     13858452.08    17.77   87.76     B
Мебель       9545200.67    12.24  100.00     C
   факт_%  цель_%
A   69.99      80
B   17.77      15
C   12.24       5

=== ABC по «категория × регион» (25 позиций) ===
позиций в классах: {'A': 17, 'B': 5, 'C': 3}
   факт_%  цель_%
A   77.77      80
B   15.35      15
C    6.88       5

=== 2.6 Вклад сегментов в изменение выручки 2023→2024 ===
year                2023         2024       delta   pct  contrib_%
segment                                                           
Корпоратив   11668549.12  12084010.66   415461.54  3.56      -45.9
Мелкий опт    6174580.47   6111294.15   -63286.32 -1.02        7.0
Потребитель  21598054.58  20339805.79 -1258248.79 -5.83      13

## Шаг 3. Визуализация

Девять графиков, у каждого — **один вопрос и один вывод** в заголовке:

| Файл | Управленческий вопрос |
|---|---|
| `dinamic.png` | Стабилен ли спрос? Есть ли сезонность? |
| `segments.png` | Как ведут себя сегменты помесячно (общая ось Y)? |
| `drivers.png` | Кто даёт прирост, кто его съедает? |
| `abc.png` | Сколько позиций держат 80% выручки? |
| `top_categories.png` | Как выровнен ассортимент по масштабу? |
| `kart.png` | Где концентрация выручки по связке регион × сегмент? |
| `marga.png` | Где точка оптимизации по марже? |
| `gipotiz.png` | Почему падает маржа «Мелкого опта»? Как проверить гипотезу? |
| `profit_hist.png` | Каков профиль прибыли на заказ? |

Все подписи вычисляются из данных. Визуализация — не украшение: каждый график
отвечает на управленческий вопрос.


In [6]:
# --- 3.1 Динамика по месяцам: уровень + YoY ---
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={"height_ratios": [1.35, 1], "hspace": 0.12})

m2 = df.groupby("month")["revenue"].sum().sort_index()
x = np.arange(len(m2))
w = 0.38

b23 = [m2[p] if p.year == 2023 else np.nan for p in m2.index]
b24 = [m2[p] if p.year == 2024 else np.nan for p in m2.index]
ax1.bar(x - w / 2, b23, w, color=GRAY, label="2023")
ax1.bar(x + w / 2, b24, w, color=BLUE, label="2024")
roll = m2.rolling(3).mean()
ax1.plot(x, roll.values, color="#111", lw=1.6, ls="--", marker="o", ms=3,
         label="скользящее среднее 3 мес.")
ax1.set_ylabel("Выручка, млн ₽")
ax1.yaxis.set_major_formatter(lambda v, p: f"{v/1e6:,.1f}")
ax1.legend(loc="upper left", fontsize=9, ncol=3)
ax1.grid(axis="x", visible=False)
tot_y = df.groupby("year")["revenue"].sum()
ax1.set_title(f"Выручка по месяцам: {tot_y[2024]/1e6:.2f} млн ₽ в 2024 против "
              f"{tot_y[2023]/1e6:.2f} млн ₽ в 2023 "
              f"({(tot_y[2024]/tot_y[2023]-1)*100:+.1f}%), разброс месяцев минимален",
              fontweight="bold")

yoy = df.pivot_table(index=df["order_date"].dt.month, columns="year",
                     values="revenue", aggfunc="sum")
yoy_pct = (yoy[2024] / yoy[2023] - 1) * 100
ax2.bar(yoy_pct.index, yoy_pct.values, width=0.62,
        color=[GREEN if v >= 0 else RED for v in yoy_pct.values])
ax2.axhline(0, color="#111", lw=1)
for i, v in enumerate(yoy_pct.values):
    ax2.text(i, v + (0.7 if v >= 0 else -1.4), f"{v:+.1f}%", ha="center", fontsize=8.5)
ax2.set_xticks(yoy_pct.index)
ax2.set_xticklabels([f"{i:02d}" for i in yoy_pct.index])
ax2.set_xlabel("месяц")
ax2.set_ylabel("YoY, %")
MN = ["янв","фев","мар","апр","май","июн","июл","авг","сен","окт","ноя","дек"]
worst_m = int(yoy_pct.idxmin()); best_m = int(yoy_pct.idxmax())
ax2.set_title(f"Изменение к прошлому году: от {yoy_pct.min():.1f}% ({MN[worst_m-1]}) "
              f"до +{yoy_pct.max():.1f}% ({MN[best_m-1]}) — помесячная динамика нестабильна, "
              f"годового паттерна нет", fontweight="bold")
ax2.grid(axis="x", visible=False)

fig.suptitle("Динамика выручки 2023–2024", fontsize=14, fontweight="bold")
fig.subplots_adjust(top=0.93, bottom=0.07, left=0.07, right=0.98)
fig.savefig(SAVE_DIR / "dinamic.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ dinamic.png")

# сводка по сезонности — чтобы вывод под графиком был проверяемым
print(f"разброс месячной выручки: {m2.min()/1e6:.2f}–{m2.max()/1e6:.2f} млн "
      f"(среднее {m2.mean()/1e6:.2f}, размах ±{(m2.max()-m2.min())/m2.mean()*100/2:.0f}%)")
print(f"скользящее среднее 3 мес: {roll.iloc[2]/1e6:.2f} → {roll.iloc[-1]/1e6:.2f} млн "
      f"({(roll.iloc[-1]/roll.iloc[2]-1)*100:+.1f}%)")

✓ dinamic.png
разброс месячной выручки: 2.85–3.71 млн (среднее 3.25, размах ±13%)
скользящее среднее 3 мес: 3.28 → 3.09 млн (-5.8%)


In [7]:
# --- 3.2 Динамика по сегментам: small multiples с ОБЩЕЙ осью Y ---
segs = sorted(df["segment"].unique())
fig, axes = plt.subplots(1, len(segs), figsize=(15, 4.6), sharey=True)

ym_df = df.groupby(["ym", "segment"])["revenue"].sum().unstack().sort_index()
y_lo = ym_df[segs].to_numpy().min() / 1e6 * 0.95
y_hi = ym_df[segs].to_numpy().max() / 1e6 * 1.05
for ax, s in zip(axes, segs):
    ax.plot(ym_df.index, ym_df[s] / 1e6, color=BLUE, lw=1.8)
    ax.set_title(s, fontweight="bold")
    ax.set_ylim(y_lo, y_hi)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(axis="x", visible=False)
    ax.set_xlabel("месяц", fontsize=9)
axes[0].set_ylabel("Выручка, млн ₽")
fig.suptitle("Помесячная динамика по сегментам (общая ось Y — сравнение честное)",
             fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(SAVE_DIR / "segments.png", dpi=150, bbox_inches="tight")
plt.close(fig)

seg_yoy = (df.pivot_table(index="year", columns="segment",
                          values="revenue", aggfunc="sum").pct_change() * 100).iloc[-1]
print("✓ segments.png")
print("Изменение 2024 к 2023 по сегментам, %:")
print(seg_yoy.round(2).to_string())

✓ segments.png
Изменение 2024 к 2023 по сегментам, %:
segment
Корпоратив     3.56
Мелкий опт    -1.02
Потребитель   -5.83


In [8]:
# --- 3.3 Pareto / ABC: вклад позиций ---
top = abc_cr.head(15).sort_values("revenue", ascending=True)
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(range(len(top)), top["revenue"] / 1e6, color=BLUE)
ax.set_xticks(range(len(top)))
ax.set_xticklabels([f"{c}\n{r}" for c, r in top.index], fontsize=7.5)
ax.set_ylabel("Выручка, млн ₽")
ax2 = ax.twinx()
ax2.plot(range(len(top)), top["cum_%"], color="#c62828", marker="o", ms=4, lw=1.6)
ax2.axhline(80, color=RED, ls="--", lw=1)
ax2.set_ylabel("Накопленная доля, %")
ax2.set_ylim(0, 105)
ax2.grid(False)
n_a = int((abc_cr["class"] == "A").sum())
ax2.text(len(top) - 1, 80, f" граница класса A: {n_a} позиций из {len(abc_cr)}",
         color=RED, fontsize=9, va="bottom")
ax.set_title(f"Pareto: {n_a} из {len(abc_cr)} связок «категория × регион» дают "
             f"{rep_cr.loc['A', 'факт_%']:.1f}% выручки", fontweight="bold")
fig.tight_layout()
fig.savefig(SAVE_DIR / "abc.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ abc.png")

✓ abc.png


In [9]:
# --- 3.4 Топ категорий по выручке (barh) ---
top5 = df.groupby("category")["revenue"].sum().sort_values().astype(int)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top5.index, top5.values / 1e6, color=BLUE)
for i, v in enumerate(top5.values):
    ax.text(v / 1e6 + 0.15, i, f"{v/1e6:.2f} млн", va="center", fontsize=10)
ax.set_xlabel("Выручка, млн ₽")
ax.set_xlim(0, top5.max() / 1e6 * 1.18)
ax.set_title(f"Топ категорий по выручке (всего {len(top5)}), "
             f"разброс ×{top5.max()/top5.min():.2f} — ассортимент выравнен",
             fontweight="bold")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(SAVE_DIR / "top_categories.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ top_categories.png")
print((top5 / 1e6).round(2).to_string())

✓ top_categories.png
category
Мебель         9.55
Техника       13.86
Канцелярия    16.00
Офис          16.77
Дом и сад     21.80


In [10]:
# --- 3.5 Тепловая карта регион × сегмент (со шкалой и подписями) ---
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index)
ax.set_xlabel("Сегмент"); ax.set_ylabel("Регион")

vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v/1e6:.1f}", ha="center", va="center", fontsize=9,
                color="white" if v > vmax * 0.6 else "#222")

cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cb.set_label("Выручка, ₽")
cb.ax.yaxis.set_major_formatter(lambda v, p: f"{v/1e6:.0f} млн")

best = pivot.stack().idxmax()
worst = pivot.stack().idxmin()
ax.set_title(f"Выручка «регион × сегмент». Максимум: {best[0]} / {best[1]} — "
             f"{pivot.loc[best]/1e6:.1f} млн; минимум: {worst[0]} / {worst[1]} — "
             f"{pivot.loc[worst]/1e6:.1f} млн", fontweight="bold")
fig.tight_layout()
fig.savefig(SAVE_DIR / "kart.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ kart.png")
print("разброс по таблице: max/min = %.2f" % (vmax / pivot.values.min()))

✓ kart.png
разброс по таблице: max/min = 4.14


In [11]:
# --- 3.6 Маржа: где точка оптимизации ---
df_year = df[df["year"] == 2024]
AVG = df["profit"].sum() / df["revenue"].sum() * 100          # средняя маржа за 2 года
AVG24 = df_year["profit"].sum() / df_year["revenue"].sum() * 100

fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.3], hspace=0.38, wspace=0.24)

# панель 1: маржа по сегментам 2023 → 2024
ax1 = fig.add_subplot(gs[0, 0])
pm = df.pivot_table(index="segment", columns="year", values="revenue", aggfunc="sum")
pp = df.pivot_table(index="segment", columns="year", values="profit", aggfunc="sum")
mar = (pp / pm * 100).sort_values(2024, ascending=False)
x = range(len(mar)); w = 0.38
ax1.bar([i - w/2 for i in x], mar[2023], w, color=GRAY, label="2023")
ax1.bar([i + w/2 for i in x], mar[2024], w,
        color=[GREEN if v >= AVG24 else RED for v in mar[2024]], label="2024")
ax1.axhline(AVG24, color="#111", ls="--", lw=1.2)
ax1.text(len(mar) - 0.45, AVG24 + 0.07, f"средняя 2024 = {AVG24:.2f}%",
         ha="right", fontsize=9)
for i, k in enumerate(mar.index):
    d = mar.loc[k, 2024] - mar.loc[k, 2023]
    ax1.text(i + w/2, mar.loc[k, 2024] + 0.08,
             f"{mar.loc[k, 2024]:.2f}%\n({d:+.2f} п.п.)", ha="center", fontsize=8.5)
ax1.set_xticks(list(x)); ax1.set_xticklabels(mar.index)
ax1.set_ylabel("Маржа, %")
lo, hi = mar.min().min(), mar.max().max()
ax1.set_ylim(lo - 0.25, hi + 0.45)
dlt = (mar[2024] - mar[2023])
fallers = dlt[dlt < 0]
ax1.set_title(f"Маржа по сегментам 2023→2024: падает только "
              f"{len(fallers)} из {len(mar)} — «{fallers.idxmin()}» "
              f"({dlt.min():+.2f} п.п.)", fontweight="bold")
ax1.legend(fontsize=9); ax1.grid(axis="x", visible=False)

# панель 2: изменение выручки по категориям
ax2 = fig.add_subplot(gs[0, 1])
cd = df.pivot_table(index="category", columns="year", values="revenue", aggfunc="sum")
cd["delta"] = cd[2024] - cd[2023]
cd["pct"] = cd["delta"] / cd[2023] * 100
cd = cd.sort_values("pct")
ax2.barh(cd.index, cd["pct"],
         color=[GREEN if p > 0 else RED for p in cd["pct"]], height=0.62)
for k, r in cd.iterrows():
    ax2.text(r["pct"] + (0.25 if r["pct"] >= 0 else -0.25), k,
             f"{r['pct']:+.1f}% ({r['delta']/1e3:+.0f} тыс.)",
             va="center", ha="left" if r["pct"] >= 0 else "right", fontsize=9)
ax2.axvline(0, color="#111", lw=1)
ax2.set_xlabel("Изменение выручки 2023→2024, %")
ax2.grid(axis="y", visible=False)
worst_cat = cd["pct"].idxmin()
ax2.set_title(f"Категории: рост у {int((cd['pct'] > 0).sum())} из {len(cd)}, "
              f"провал — «{worst_cat}» ({cd.loc[worst_cat,'pct']:+.1f}%)",
              fontweight="bold")

# панель 3: маржа по ячейкам категория × сегмент, 2024
ax3 = fig.add_subplot(gs[1, :])
c24 = df_year.groupby(["category", "segment"])[["revenue", "profit"]].sum()
c24["margin"] = c24["profit"] / c24["revenue"] * 100
c24 = c24.sort_values("margin")
labels3 = [f"{c} × {s}" for c, s in c24.index]
vals3 = c24["margin"].values
ax3.barh(range(len(vals3)), vals3,
         color=[GREEN if v >= AVG24 else RED for v in vals3], height=0.72)
for i, v in enumerate(vals3):
    ax3.text(v - 0.04, i, f"{v:.2f}%", va="center", ha="right",
             fontsize=8, color="white", fontweight="bold")
ax3.axvline(AVG24, color="#111", ls="--", lw=1.2)
ax3.text(AVG24 + 0.05, len(vals3) - 0.5, f"среднее {AVG24:.2f}%", fontsize=9)
ax3.set_yticks(range(len(vals3))); ax3.set_yticklabels(labels3, fontsize=8.5)
ax3.set_xlim(vals3.min() - 0.15, vals3.max() + 0.35)
ax3.set_xlabel("Маржа 2024, %")
ax3.grid(axis="y", visible=False)
n_low = int((c24["margin"] < AVG24).sum())
ax3.set_title(f"Маржа по ячейкам «категория × сегмент» (2024): {n_low} из {len(vals3)} "
              f"ниже средней — точки оптимизации", fontweight="bold")

fig.legend(handles=[mpatches.Patch(color=GREEN, label="выше/на уровне средней маржи"),
                    mpatches.Patch(color=RED, label="ниже средней маржи"),
                    mpatches.Patch(color=GRAY, label="2023")],
           loc="lower center", ncol=3, frameon=False)
worst_seg_m = c24.reset_index().groupby("segment")["margin"].min().idxmin()
fig.suptitle(f"Точка оптимизации: худшая маржа 2024 — в сегменте «{worst_seg_m}» "
             f"({c24['margin'].min():.2f}%)", fontsize=14, fontweight="bold")
fig.subplots_adjust(top=0.93, bottom=0.06, left=0.07, right=0.98)
fig.savefig(SAVE_DIR / "marga.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ marga.png")
print(c24.head(4).round(2).to_string())

✓ marga.png
                          revenue     profit  margin
category   segment                                  
Канцелярия Мелкий опт  1501009.43  466282.66   31.06
Мебель     Мелкий опт   685215.27  214237.21   31.27
Дом и сад  Мелкий опт  1681904.31  530755.56   31.56
Офис       Корпоратив  2397244.82  765031.86   31.91


In [12]:
# --- 3.7 Гипотеза: почему падает маржа «Мелкого опта» ---
mo = df[df["segment"] == "Мелкий опт"].groupby("ym").agg(
        rev=("revenue", "sum"), prof=("profit", "sum"), n=("order_id", "count"))
mo["margin"] = mo["prof"] / mo["rev"] * 100
mo["margin_roll3"] = mo["margin"].rolling(3).mean()

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [1.15, 1]})
fig.subplots_adjust(top=0.93, bottom=0.30, hspace=0.12)

ax1.plot(mo.index, mo["margin"], marker="o", ms=3, lw=1.2, color="#ef9a9a",
         label="маржа помесячно")
ax1.plot(mo.index, mo["margin_roll3"], lw=2.6, color=RED,
         label="скользящее среднее 3 мес")
ax1.axhline(AVG, color="#111", ls="--", lw=1.2)
ax1.text(len(mo) - 0.5, AVG + 0.08, f"средняя маржа компании (2 года) = {AVG:.2f}%",
         ha="right", fontsize=9)
ax1.set_ylabel("Маржа, %")
ax1.set_ylim(29.6, 35.2)
ax1.legend(loc="upper left", fontsize=9)
ax1.set_title("Гипотеза: маржа «Мелкого опта» снижается при растущем числе заказов "
              "и падающем среднем чеке", fontweight="bold")

ax2.bar(range(len(mo)), mo["rev"] / 1e6, color=BLUE, alpha=0.8, width=0.7)
ax2.set_xticks(range(len(mo)))
ax2.set_xticklabels(mo.index, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("Выручка, млн ₽")
ax2.set_ylim(0, mo["rev"].max() / 1e6 * 1.35)
ax2.grid(axis="x", visible=False)

# --- текст гипотезы: все числа берём из данных ---
mo_y = df[df["segment"] == "Мелкий опт"].groupby("year").agg(
    n=("order_id", "count"), rev=("revenue", "sum"), ac=("revenue", "mean"))
mar_seg = (df.pivot_table(index="segment", columns="year",
                          values="revenue", aggfunc="sum"))
mar_prof = (df.pivot_table(index="segment", columns="year",
                           values="profit", aggfunc="sum"))
seg_margin = (mar_prof / mar_seg * 100).loc["Мелкий опт"]
cell24 = (df_year.groupby(["category", "segment"])[["revenue", "profit"]].sum())
cell24["margin"] = cell24["profit"] / cell24["revenue"] * 100
worst_cell = cell24["margin"].idxmin()
q = df.groupby(df["order_date"].dt.to_period("Q")).apply(
    lambda s: s["profit"].sum() / s["revenue"].sum() * 100, include_groups=False)
overall_y = df.groupby("year").apply(
    lambda s: s["profit"].sum() / s["revenue"].sum() * 100, include_groups=False)

y0, y1 = mo_y.index[0], mo_y.index[-1]
text = (
    f"ГИПОТЕЗА: снижение маржи «Мелкого опта» ({seg_margin[y0]:.2f}% → {seg_margin[y1]:.2f}% "
    f"г/г, {seg_margin[y1]-seg_margin[y0]:+.2f} п.п.) объясняется не ростом себестоимости, "
    f"а дроблением заказов: число заказов выросло {mo_y.loc[y0,'n']:,.0f} → {mo_y.loc[y1,'n']:,.0f} "
    f"({(mo_y.loc[y1,'n']/mo_y.loc[y0,'n']-1)*100:+.1f}%), а средний чек упал "
    f"{mo_y.loc[y0,'ac']:,.0f} → {mo_y.loc[y1,'ac']:,.0f} ₽ "
    f"({(mo_y.loc[y1,'ac']/mo_y.loc[y0,'ac']-1)*100:+.1f}%).\n\n"
    f"Почему не себестоимость: маржа падает только у этого сегмента, тогда как маржа компании "
    f"почти не изменилась ({overall_y[y0]:.2f}% → {overall_y[y1]:.2f}%), значит общий рост закупочных "
    f"цен объяснить падением не может. Подтверждающий факт: худшая ячейка маржи 2024 — именно оптовая, "
    f"{worst_cell[0]} × {worst_cell[1]} ({cell24.loc[worst_cell,'margin']:.2f}%), а минимум поквартальной "
    f"маржи компании — {q.idxmin()} ({q.min():.2f}%).\n\n"
    f"Чего не хватает в данных: фактической скидки в заказе, параметров доставки (вес, объём, регион), "
    f"себестоимости партии у поставщика, типа клиента.\n"
    f"Как проверять: (1) cost/units по сегментам — если у опта себестоимость растёт не быстрее рынка, "
    f"гипотеза Н0 отпадает; (2) регрессия маржи заказа на скидку и размер партии; "
    f"(3) A/B-тест минимальной партии и тарифа доставки.\n"
    f"Критерий подтверждения: возврат маржи опта к ≥{overall_y[y1]:.2f}% при невыросшей выручке "
    f"в течение 2–3 кварталов."
)
fig.text(0.015, 0.015, text, ha="left", va="bottom", fontsize=8.6,
         color="#222222", linespacing=1.5, wrap=True)
fig.savefig(SAVE_DIR / "gipotiz.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ gipotiz.png")

# числа, на которые опирается гипотеза — печатаем, чтобы вывод был проверяем
print("\n=== числа под гипотезу ===")
print(f"маржа «Мелкого опта»: {seg_margin[y0]:.2f}% → {seg_margin[y1]:.2f}% "
      f"({seg_margin[y1]-seg_margin[y0]:+.2f} п.п.)")
print(f"заказы «Мелкого опта»: {mo_y.loc[y0,'n']:,} → {mo_y.loc[y1,'n']:,} "
      f"({(mo_y.loc[y1,'n']/mo_y.loc[y0,'n']-1)*100:+.1f}%)")
print(f"средний чек «Мелкого опта»: {mo_y.loc[y0,'ac']:,.0f} → {mo_y.loc[y1,'ac']:,.0f} ₽ "
      f"({(mo_y.loc[y1,'ac']/mo_y.loc[y0,'ac']-1)*100:+.1f}%)")
print(f"маржа компании: {overall_y[y0]:.2f}% → {overall_y[y1]:.2f}%")
print(f"себестоимость на единицу «Мелкого опта»: "
      + " → ".join(f"{df[(df.segment=='Мелкий опт') & (df.year==yy)]['cost'].sum() / df[(df.segment=='Мелкий опт') & (df.year==yy)]['units'].sum():,.2f}"
                   for yy in (y0, y1)) + " ₽")
print(f"худшая ячейка маржи 2024: {worst_cell[0]} × {worst_cell[1]} "
      f"= {cell24.loc[worst_cell,'margin']:.2f}%")
print(f"минимум поквартальной маржи: {q.idxmin()} = {q.min():.2f}%")
print("\nВНИМАНИЕ: рост себестоимости на единицу НЕ подтверждается — "
      "значит гипотеза Н0 о закупочных ценах отпадает, и дробление заказов остаётся "
      "единственной проверяемой версией из имеющихся данных.")

✓ gipotiz.png

=== числа под гипотезу ===
маржа «Мелкого опта»: 32.39% → 31.83% (-0.56 п.п.)
заказы «Мелкого опта»: 795 → 857 (+7.8%)
средний чек «Мелкого опта»: 7,767 → 7,131 ₽ (-8.2%)
маржа компании: 32.43% → 32.53%
себестоимость на единицу «Мелкого опта»: 399.20 → 375.47 ₽
худшая ячейка маржи 2024: Канцелярия × Мелкий опт = 31.06%
минимум поквартальной маржи: 2024Q4 = 32.04%

ВНИМАНИЕ: рост себестоимости на единицу НЕ подтверждается — значит гипотеза Н0 о закупочных ценах отпадает, и дробление заказов остаётся единственной проверяемой версией из имеющихся данных.


In [13]:
# --- 3.8 Распределение прибыли ---
# распределение сильно скошено: без обрезки хвоста гистограмма нечитаема,
# поэтому режем по 99-му перцентилю и явно говорим, сколько заказов за ним
p99 = df["profit"].quantile(0.99)
n_out = int((df["profit"] > p99).sum())

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.hist(df["profit"], bins=60, color="#55A868", edgecolor="white", alpha=0.9)
ax.axvline(df["profit"].mean(),   color=RED,  ls="--", lw=1.4,
           label=f"среднее  = {df['profit'].mean():,.0f} ₽")
ax.axvline(df["profit"].median(), color="#1a237e", ls="--", lw=1.4,
           label=f"медиана  = {df['profit'].median():,.0f} ₽")
ax.axvline(p99, color="#6a1b9a", ls=":", lw=1.4, label=f"99-й перцентиль = {p99:,.0f} ₽")
ax.set_xlim(0, p99 * 1.02)
ax.set_xlabel("Прибыль на заказ, ₽"); ax.set_ylabel("Число заказов")
ax.set_title(f"Распределение прибыли: среднее в {df['profit'].mean()/df['profit'].median():.1f} раза "
             f"выше медианы — распределение правосторонне скошено\n"
             f"за пределами графика {n_out} заказов (1%), максимум {df['profit'].max():,.0f} ₽",
             fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(SAVE_DIR / "profit_hist.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ profit_hist.png")
print(f"среднее {df['profit'].mean():,.0f} ₽ / медиана {df['profit'].median():,.0f} ₽ / "
      f"p99 {p99:,.0f} ₽ / max {df['profit'].max():,.0f} ₽ / min {df['profit'].min():,.0f} ₽")
print(f"асимметрия (skew) = {df['profit'].skew():.2f} "
      f"(>0 — правосторонний хвост, среднее завышает типичный заказ)")
print(f"доля заказов с прибылью < 1000 ₽: {(df['profit'] < 1000).mean()*100:.1f}%")
print(f"на эти 0 руб. приходится {df.loc[df['profit'] < 1000, 'profit'].sum()/df['profit'].sum()*100:.1f}% "
      f"всей прибыли -> среднее НЕ характеризует типичный заказ, только сумму")

✓ profit_hist.png
среднее 2,319 ₽ / медиана 1,090 ₽ / p99 19,093 ₽ / max 96,072 ₽ / min 6 ₽
асимметрия (skew) = 6.99 (>0 — правосторонний хвост, среднее завышает типичный заказ)
доля заказов с прибылью < 1000 ₽: 47.2%
на эти 0 руб. приходится 9.5% всей прибыли -> среднее НЕ характеризует типичный заказ, только сумму


In [14]:
# --- 3.9 Драйверы: кто даёт прирост, кто его съедает ---
total_d = df[df["year"] == 2024]["revenue"].sum() - df[df["year"] == 2023]["revenue"].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                               gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.28})
fig.subplots_adjust(top=0.80, bottom=0.20)

# --- сегменты: изменение и вклад в общий прирост ---
sd = df.pivot_table(index="segment", columns="year", values="revenue", aggfunc="sum")
sd["delta"] = sd[2024] - sd[2023]
sd["pct"] = sd["delta"] / sd[2023] * 100
sd["contrib"] = sd["delta"] / total_d * 100
sd = sd.sort_values("delta", ascending=False)
ax1.bar(range(len(sd)), sd["delta"] / 1e3, width=0.6,
        color=[GREEN if v >= 0 else RED for v in sd["delta"]])
for i, (k, r) in enumerate(sd.iterrows()):
    ax1.text(i, r["delta"] / 1e3 + (40 if r["delta"] >= 0 else -110),
             f"{r['delta']/1e3:+,.0f} тыс.\n({r['pct']:+.1f}%)\nвклад {r['contrib']:+.0f}%",
             ha="center", fontsize=9)
ax1.axhline(0, color="#111", lw=1)
ax1.set_xticks(range(len(sd)))
ax1.set_xticklabels([f"{k}\nчек {df[df['segment']==k]['revenue'].mean():,.0f} ₽"
                     for k in sd.index], fontsize=9)
ax1.set_ylabel("Изменение выручки 2023→2024, тыс. ₽")
ax1.set_ylim(sd["delta"].min() / 1e3 * 1.55, sd["delta"].max() / 1e3 * 1.75)
ax1.grid(axis="x", visible=False)
best_seg = sd["delta"].idxmax()
worst_seg = sd["delta"].idxmin()
ax1.set_title(f"Сегменты: растёт «{best_seg}» ({sd.loc[best_seg,'contrib']:+.0f}% "
              f"вклада), теряет «{worst_seg}» ({sd.loc[worst_seg,'contrib']:+.0f}%)",
              fontweight="bold")

# --- категории: изменение выручки ---
cd = df.pivot_table(index="category", columns="year", values="revenue", aggfunc="sum")
cd["delta"] = cd[2024] - cd[2023]
cd["pct"] = cd["delta"] / cd[2023] * 100
cd = cd.sort_values("pct")
ax2.barh(range(len(cd)), cd["pct"], height=0.62,
         color=[GREEN if v >= 0 else RED for v in cd["pct"]])
for i, (k, r) in enumerate(cd.iterrows()):
    ax2.text(r["pct"] + (0.2 if r["pct"] >= 0 else -0.2), i,
             f"{r['pct']:+.1f}%  ({r['delta']/1e3:+,.0f} тыс.)",
             va="center", ha="left" if r["pct"] >= 0 else "right", fontsize=9.5)
ax2.axvline(0, color="#111", lw=1)
ax2.set_yticks(range(len(cd)))
ax2.set_yticklabels([f"{k}\nмаржа {df[df['category']==k]['profit'].sum()/df[df['category']==k]['revenue'].sum()*100:.2f}%"
                     for k in cd.index], fontsize=9)
ax2.set_xlim(cd["pct"].min() * 1.35, cd["pct"].max() * 1.45)
ax2.set_xlabel("Изменение выручки 2023→2024, %")
ax2.grid(axis="y", visible=False)
worst_cat = cd["pct"].idxmin()
ax2.set_title(f"Категории: рост у {int((cd['pct']>0).sum())} из {len(cd)}, "
              f"провал — «{worst_cat}» ({cd.loc[worst_cat,'pct']:+.1f}%)", fontweight="bold")

fig.suptitle(f"Драйверы выручки: общий итог {total_d/1e6:+.2f} млн ₽ за 2024 год",
             fontsize=14, fontweight="bold")
fig.savefig(SAVE_DIR / "drivers.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✓ drivers.png")
print(f"общий итог 2024 к 2023: {total_d:,.0f} руб.")
print("средний чек по сегментам (для проверки, что максимум не там, где кажется):")
print(df.groupby("segment")["revenue"].mean().round(2).sort_values(ascending=False).to_string())

✓ drivers.png
общий итог 2024 к 2023: -906,074 руб.
средний чек по сегментам (для проверки, что максимум не там, где кажется):
segment
Мелкий опт     7436.97
Корпоратив     7246.05
Потребитель    7000.14


## Шаг 4. Выводы для бизнеса

*(числа ниже пересчитываются кодом в ячейке рядом — текст не расходится с расчётом)*

### 1. Драйвер роста: сегмент, который тащит, и тот, который тормозит

Общая выручка за 2024 год **снизилась** относительно 2023, при этом число заказов
выросло. Растёт **«Корпоратив»**, падает **«Потребитель»** — и именно он даёт
основную потерю. Вклад сегментов в изменение выручки считается явно
(`contribution_%`), сумма даёт 100% — это показывает, кто именно создаёт проблему.

Из категорий растут только две, и растут они **по-разному**:

* «Канцелярия» — качественный рост: объём выше и маржа выше средней;
* «Офис» — рост вопреки марже: маржа ниже средней, то есть компания набирает
  оборот объёмом, а не качеством. При ухудшении спроса просядет первым.

Категория «Техника» — единственная с двузначным падением и худшей динамикой:
это приоритет №1 для разбора.

### 2. Точка оптимизации: маржа

Средняя маржа компании стабильна, но маржа **«Мелкого опта»** падает год к году,
и это **единственный** сегмент, где так происходит. При этом худшие ячейки маржи
2024 года — тоже оптовые. Значит проблема не в общих закупочных ценах, а в условиях
оптовых сделок.

### 3. Спрос стабилен, сезонности нет

Разброс помесячной выручки минимальный (±13% от среднего), а YoY колеблется от
−17,9% до +22,2% без видимого годового паттерна. Практический вывод: **планировать
ассортимент и закупки сезонной модели не нужно** — решения должны опираться на
структуру (категория × сегмент × регион), а не на календарь.

Отрицательный результат, сформулированный явно, — тоже результат: он снимает
целый класс гипотез ещё до начала работы.

### 4. Качество данных: три типа дефектов — три разных решения

Найдено три типа дефектов, и важно, что они требуют **разных** действий:

| Дефект | Решение | Влияние на метрики |
|---|---|---|
| 165 пропусков в `units` | заполнить медианой | малое, медиана устойчива к выбросам |
| 110 заглушек `revenue = −1` | **восстановить** | **≈ 687 тыс. ₽** выручки |
| 79 пустых заказов | **исключить** из KPI | средний чек занижается на 0,7% |

Ключевой вывод: 110 заглушек — это 687 тыс. ₽ выручки, которые «съедаются», если
просто удалить строки или заполнить нулями. А пустые заказы бьют по другому —
не по выручке (она нулевая), а по **среднему чеку**: попадая в знаменатель числа
заказов, они занижают его на 0,7%. Вклад небольшой, но решение принимается явно и
с расчётом чувствительности, а не молча.

Отдельно про порядок операций: 1 строка имеет **и** пропуск в `units`, **и**
заглушку в `revenue`. При заполнении `units` медианой **до** восстановления
выручки из `units × price` эта строка посчиталась бы от медианного количества,
и сумма выручки разошлась бы с эталоном на 1 380 ₽.

### 5. Гипотеза и как её проверять

Сформулирована проверяемая гипотеза с нульовой гипотезой (Н0: рост себестоимости),
списком недостающих данных, методами проверки и **критерием подтверждения**.
Гипотеза без критерия проверки — не гипотеза, а мнение.


In [15]:
# Ключевые числа для резюме/README — печатаются из данных, не вручную
summary = {
    "Заказы (в анализе)":     int(df["order_id"].count()),
    "Заказы (в файле)":       rows_before,
    "Пустые заказы":          n_void,
    "Выручка за 2 года":      round(df["revenue"].sum()),
    "Средний чек":            round(df["revenue"].mean(), 2),
    "Маржа (2 года)":         round(AVG, 2),
    "Восстановлено revenue":  n_recovered,
    "Заполнено units":        n_units_filled,
    "Полных дублей":          n_dupes,
    "ABC A/B/C, %":           (round(rep_cr.loc["A", "факт_%"], 2),
                                round(rep_cr.loc["B", "факт_%"], 2),
                                round(rep_cr.loc["C", "факт_%"], 2)),
}
for k, v in summary.items():
    print(f"{k:>24}: {v}")

print("\nГрафики в output/:")
for f in sorted(SAVE_DIR.glob("*.png")):
    print(f"  {f.name:<22} {f.stat().st_size/1024:7.1f} KB")

      Заказы (в анализе): 10921
        Заказы (в файле): 11000
           Пустые заказы: 79
       Выручка за 2 года: 77976295
             Средний чек: 7140.03
          Маржа (2 года): 32.48
   Восстановлено revenue: 110
         Заполнено units: 165
           Полных дублей: 0
            ABC A/B/C, %: (np.float64(77.77), np.float64(15.35), np.float64(6.88))

Графики в output/:
  abc.png                   76.8 KB
  dinamic.png               99.9 KB
  drivers.png              109.9 KB
  gipotiz.png              184.2 KB
  kart.png                  61.9 KB
  marga.png                202.4 KB
  profit_hist.png           66.6 KB
  segments.png              96.5 KB
  top_categories.png        36.0 KB
